# 20 · Event effects, randomization inference, joint estimation and hazard models

Inputs: `panel_event`, `panel_cem_strata` (05)

Outputs: `est_20_effects.csv.gz`, `est_20_pooled.csv`, `est_20_joint.csv`, `est_20_hazard.csv`, `est_20_paths.csv.gz`; `fig_20_effects.png`, `fig_20_paths.png`, `fig_20_km.png`, `fig_20_hetero.png`; `build_report_20.json`

1. θ_d: CEM-weighted treated − control difference in the cumulative share at relative day r, for each event, placebo, negative-control and positive-control date (key-weighted main, BTC-weighted secondary). Controls are reweighted to the treated cell distribution within common-support cells.
2. Baseline: local linear kernel regression of placebo θ on calendar date (Gaussian kernel, h = 90 days; 60 and 120 for robustness); β_e = θ_e − f̂(t_e). Placebo pools: main pool of `placebo_draws_strict` (29 dates); symmetric ±30 pool (15 dates) for robustness. FTX (2022-11) and Randstorm (2023-11) precede the pool (from 2024-06) and use the pool mean.
3. Inference: leave-one-out placebo residuals θ_p − f̂₋ₚ(t_p) form the null. Single-event p = (1 + #{|residual| ≥ |β_e|}) / (1 + pool size). Pooled test of the 6 main events: mean β against the mean residuals of the 1,000 draws of 6 placebo dates. Sampling standard errors from within-cell sums of squares are secondary.
4. Contrasts: `LC` (L vs C, six-dimensional cells; main), `TC` (T vs C, five-dimensional cells without script family), `RS` (Randstorm: first seen 2011–15 vs 2016–19, L and C combined, five-dimensional cells without first-seen year, keys with an incoming transfer in the previous 12 months), `RS_all` (no inflow restriction). All contrasts are computed on all dates.


In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"
AUTO_RUN_USD, PRICE_PER_TIB_USD = 5.0, 6.25

H_MAIN      = 90                     # local linear bandwidth (days, standard deviation of the Gaussian kernel)
BANDWIDTHS  = [60, 90, 120]
POOLS       = ["main", "sym30"]      # the two pools of placebo_draws_strict; main is primary
REPORT_DAYS = (7, 30)
JOINT       = ("google", "bip361")   # the two jointly estimated events
SEGMENTS    = [(0, 7), (8, 30)]
OUT_C = ["y1_strict", "y1_mid", "y1_loose", "y2", "y0", "y3", "y1_strict_held", "y3_held"]   # cumulative shares
OUT_D = ["y1_strict", "y1_strict_held", "y3", "y0"]                                          # daily shares (joint estimation)
HZ    = ["y1s_50", "y1s_gt0", "y1sh_50", "y0_50"]                                            # hazard events
HR_INTERVALS = {"0-7": (0, 7), "8-30": (8, 30), "0-30": (0, 30)}
CONTRASTS = ["LC", "TC", "RS", "RS_all"]
N_PAIRS_MAX = 2000

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass
from google.cloud import bigquery
import json, os, datetime as dt
import numpy as np
import pandas as pd
from scipy.optimize import brentq

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"
SPENT = {"usd": 0.0}

def q(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    usd = job.total_bytes_processed / 2**40 * PRICE_PER_TIB_USD
    assert usd < AUTO_RUN_USD, f"estimated ${usd:.2f}, above the auto-run limit"
    SPENT["usd"] += usd
    return client.query(sql).to_dataframe()

for t in ["panel_event", "panel_cem_strata", "placebo_draws_strict"]:
    client.get_table(f"{DS}.{t}")
print("Tables from 05 found")

## Step 1 · CEM differences θ per date × relative day

One row per event × relative day × contrast × variant (all / no_lm) × subgroup (all / balance strata). Per outcome k: treated sum `T_k`, reweighted control sum `C_k`, within-cell variance terms `vT_k`, `vC_k`, and BTC-weighted sums `bT_k`, `bC_k`. Mean = sum / `den` (BTC-weighted: / `bden`).

Outcomes: `c:*` cumulative shares, `d:*` daily shares, `h:*` cumulative incidence of first de-risking (key-weighted only).

In [ ]:
OUTS = ([(f"c_{o}", f"{o}_c", f"{o}_c2", f"{o}_cb") for o in OUT_C] +
        [(f"d_{o}", f"{o}_d", f"{o}_d2", f"{o}_db") for o in OUT_D] +
        [(f"h_{h}", f"h_{h}_cum", f"h_{h}_cum", "0") for h in HZ])
CELL = "PARTITION BY event, grp, cem_id, is_large_mover ORDER BY rel_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW"
NF  = "CONCAT(bal_bin, '|', age_bin, '|', fs_bin, '|', rec_bin, '|', nutxo_bin)"
NFS = "CONCAT(bal_bin, '|', age_bin, '|', family, '|', rec_bin, '|', nutxo_bin)"
RS_ARM = "CASE fs_bin WHEN '2011-15' THEN 'T' WHEN '2016-19' THEN 'C' END"
UNNEST_CONTRASTS = f'''UNNEST([
    STRUCT('LC' AS contrast, CASE grp WHEN 'L' THEN 'T' WHEN 'C' THEN 'C' END AS arm, cem_id AS stratum),
    STRUCT('TC', CASE grp WHEN 'T' THEN 'T' WHEN 'C' THEN 'C' END, {NF}),
    STRUCT('RS', IF(grp != 'T' AND rec_bin != '365d+', {RS_ARM}, NULL), {NFS}),
    STRUCT('RS_all', IF(grp != 'T', {RS_ARM}, NULL), {NFS})
  ]) AS x'''
DIMCOLS = "event, grp, cem_id, is_large_mover, rel_day, bal_bin, age_bin, family, fs_bin, rec_bin, nutxo_bin, n_keys, bal_btc"

pe_cols = ",\n    ".join(f"{a} AS {k}_c, {b} AS {k}_c2, {c} AS {k}_cb" for k, a, b, c in OUTS if not k.startswith("h_"))
pe_hz   = ",\n    ".join(f"SUM(h_{h}_n) OVER ({CELL}) AS h_{h}_cum" for h in HZ)
pe_hcol = ",\n    ".join(f"h_{h}_cum AS h_{h}_c, h_{h}_cum AS h_{h}_c2, 0 AS h_{h}_cb" for h in HZ)
K = [k for k, *_ in OUTS]
s_sum = ",\n    ".join(f"SUM({k}_c) AS {k}_c, SUM({k}_c2) AS {k}_c2, SUM({k}_cb) AS {k}_cb" for k in K)
p_piv = ",\n    ".join(
    f"SUM(IF(arm = 'T', {k}_c, 0)) AS cT_{k}, SUM(IF(arm = 'C', {k}_c, 0)) AS cC_{k}, "
    f"SUM(IF(arm = 'T', {k}_c2, 0)) AS qT_{k}, SUM(IF(arm = 'C', {k}_c2, 0)) AS qC_{k}, "
    f"SUM(IF(arm = 'T', {k}_cb, 0)) AS bT_{k}, SUM(IF(arm = 'C', {k}_cb, 0)) AS bC_{k}" for k in K)
M = "(nT > 0 AND nC > 0)"
MB = "(nT > 0 AND nC > 0 AND bT > 0 AND bC > 0)"
f_sum = ",\n  ".join(
    f"SUM(IF({M}, cT_{k}, 0)) AS T_{k}, "
    f"SUM(IF({M}, SAFE_DIVIDE(nT, nC) * cC_{k}, 0)) AS C_{k},\n  "
    f"SUM(IF({M}, qT_{k} - SAFE_DIVIDE(cT_{k} * cT_{k}, nT), 0)) AS vT_{k}, "
    f"SUM(IF({M}, POW(SAFE_DIVIDE(nT, nC), 2) * (qC_{k} - SAFE_DIVIDE(cC_{k} * cC_{k}, nC)), 0)) AS vC_{k},\n  "
    f"SUM(IF({MB}, bT_{k}, 0)) AS bT_{k}, SUM(IF({MB}, SAFE_DIVIDE(bT, bC) * bC_{k}, 0)) AS bC_{k}" for k in K)

SQL_THETA = f'''
WITH pe0 AS (
  SELECT {DIMCOLS},
    {pe_cols},
    {pe_hz}
  FROM `{DS}.panel_event`
),
pe AS (
  SELECT pe0.*, {pe_hcol} FROM pe0
),
u AS (
  SELECT pe.*, x.contrast, x.arm, x.stratum, v AS variant
  FROM pe,
  {UNNEST_CONTRASTS},
  UNNEST(IF(pe.is_large_mover, ['all'], ['all', 'no_lm'])) AS v
  WHERE x.arm IS NOT NULL
),
s AS (
  SELECT event, rel_day, contrast, variant, stratum, arm, ANY_VALUE(bal_bin) AS bal_bin,
    SUM(n_keys) AS n, SUM(bal_btc) AS b,
    {s_sum}
  FROM u
  GROUP BY event, rel_day, contrast, variant, stratum, arm
),
p AS (
  SELECT event, rel_day, contrast, variant, stratum, ANY_VALUE(bal_bin) AS bal_bin,
    SUM(IF(arm = 'T', n, 0)) AS nT, SUM(IF(arm = 'C', n, 0)) AS nC,
    SUM(IF(arm = 'T', b, 0)) AS bT, SUM(IF(arm = 'C', b, 0)) AS bC,
    {p_piv}
  FROM s
  GROUP BY event, rel_day, contrast, variant, stratum
),
f AS (
  SELECT p.*, sg FROM p, UNNEST(['all', bal_bin]) AS sg
)
SELECT event, rel_day, contrast, variant, sg AS subgroup,
  SUM(nT) AS nT_all, SUM(nC) AS nC_all,
  SUM(IF({M}, nT, 0)) AS den, SUM(IF({M}, nC, 0)) AS denC, SUM(IF({MB}, bT, 0)) AS bden,
  {f_sum}
FROM f
GROUP BY event, rel_day, contrast, variant, subgroup
'''
th = q(SQL_THETA)
ev = q(f'''SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date,
                  ANY_VALUE(censor_day) AS censor_day, ANY_VALUE(strict_ok) AS strict_ok,
                  ANY_VALUE(cb_confounded) AS cb_confounded
           FROM `{DS}.panel_event` GROUP BY event''')
ev["event_date"] = pd.to_datetime(ev.event_date)
ev = ev.sort_values("event_date").reset_index(drop=True)
print(f"theta table {len(th):,} rows; {ev.event.nunique()} dates:", ev.event_set.value_counts().to_dict())

# long table: each row = series (contrast, variant, subgroup, outcome, weight, stat, rel_day) x event
rows = []
for k in K:
    base = th[["event", "rel_day", "contrast", "variant", "subgroup"]].copy()
    den, bden = th.den.replace(0, np.nan), th.bden.replace(0, np.nan)
    mT, mC = th[f"T_{k}"] / den, th[f"C_{k}"] / den
    se = np.sqrt((th[f"vT_{k}"] + th[f"vC_{k}"]).clip(lower=0)) / den
    for stat, val in [("diff", mT - mC), ("T", mT), ("C", mC), ("se", se)]:
        rows.append(base.assign(outcome=k, weight="key", stat=stat, value=val.values))
    if not k.startswith("h_"):
        bT, bC = th[f"bT_{k}"] / bden, th[f"bC_{k}"] / bden
        for stat, val in [("diff", bT - bC), ("T", bT), ("C", bC)]:
            rows.append(base.assign(outcome=k, weight="btc", stat=stat, value=val.values))
L = pd.concat(rows, ignore_index=True)
SER = ["contrast", "variant", "subgroup", "outcome", "weight", "stat", "rel_day"]
TH = L.pivot_table(index=SER, columns="event", values="value", aggfunc="first")
TH = TH.reindex(columns=ev.event.tolist())
print("Series:", len(TH), "; matching rate (LC, all, r=0):")
m0 = th[(th.contrast == "LC") & (th.variant == "all") & (th.subgroup == "all") & (th.rel_day == 0)].set_index("event")
display((m0.den / m0.nT_all).rename("T_matched").describe().round(3))

## Step 2 · Baseline smoothing and randomization inference

f̂(t_e) = Σ_p S[e, p] · θ_p for a fixed pool and bandwidth; for pool dates the row of S excludes the date itself (leave-one-out). Dates more than 60 days before the pool start (FTX, Randstorm) use the pool mean. S is computed per missingness pattern of pool dates.

In [ ]:
T0 = pd.Timestamp("2020-01-01")
tday = ((ev.event_date - T0).dt.days).to_numpy(float)
EV_IDX = {e: i for i, e in enumerate(ev.event)}
draws_all = q(f"SELECT pool, draw_id, slot, fake_date FROM `{DS}.placebo_draws_strict`")
draws_all["fake_date"] = pd.to_datetime(draws_all.fake_date)
pl = ev[ev.event_set == "placebo"].set_index("event_date").event
POOL_EVENTS, DRAWS = {}, {}
for pool in POOLS:
    d = draws_all[draws_all.pool == pool]
    dates = sorted(d.fake_date.unique())
    missing = [x for x in dates if x not in pl.index]
    assert not missing, f"dates of pool {pool} not found in panel_event: {missing}"
    POOL_EVENTS[pool] = [pl[x] for x in dates]
    pos = {x: j for j, x in enumerate(dates)}
    D = np.zeros((d.draw_id.nunique(), len(dates)))
    for r in d.itertuples():
        D[r.draw_id - 1, pos[r.fake_date]] += 1
    DRAWS[pool] = D / D.sum(1, keepdims=True)
    print(f"pool {pool}: {len(dates)} dates, {len(D)} draws")

def smoother(pool, h, mask=None):
    # returns S (events x pool). mask: usable pool columns
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp = tday[idx]
    ok = np.ones(len(idx), bool) if mask is None else mask
    lo = tp[ok].min() - 60
    S = np.zeros((len(ev), len(idx)))
    for i in range(len(ev)):
        w = np.exp(-0.5 * ((tp - tday[i]) / h) ** 2) * ok
        self_ = idx == i
        w[self_] = 0
        if tday[i] < lo:                           # before the pool: pool mean
            w = ok.astype(float); w[self_] = 0
            S[i] = w / w.sum(); continue
        X = np.column_stack([np.ones_like(tp), tp - tday[i]])
        A = X.T @ (X * w[:, None])
        S[i] = np.linalg.solve(A, (X * w[:, None]).T)[0]
    return S

def effects(Th, pool, h):
    # Th: series x events matrix (DataFrame). Returns beta (same shape) and leave-one-out pool residuals R (series x pool)
    idx = [EV_IDX[e] for e in POOL_EVENTS[pool]]
    Y = Th.to_numpy(float)
    P = Y[:, idx]
    B = np.full_like(Y, np.nan)
    nanmask = np.isnan(P)
    keys = pd.Series([m.tobytes() for m in nanmask])
    for key, rows_ in keys.groupby(keys).groups.items():
        rows_ = np.array(list(rows_))
        ok = ~nanmask[rows_[0]]
        if ok.sum() < 5:
            continue
        S = smoother(pool, h, ok)
        B[rows_] = Y[rows_] - np.nan_to_num(P[rows_]) @ S.T
    R = B[:, idx]
    return pd.DataFrame(B, index=Th.index, columns=Th.columns), R

def ri_p(beta, R):
    # two-sided randomization p-value per event: beta (series x events), R (series x pool)
    b = np.abs(np.asarray(beta, float))[:, :, None]
    r = np.abs(R)[:, None, :]
    cnt = np.sum(r >= b - 1e-15, axis=2)
    n = np.sum(~np.isnan(R), axis=1)[:, None]
    p = (1 + cnt) / (1 + n)
    p[np.isnan(np.asarray(beta, float))] = np.nan
    return p

def ri_ci(beta, R, a=0.10):
    lo_q, hi_q = np.nanquantile(R, 1 - a / 2, axis=1), np.nanquantile(R, a / 2, axis=1)
    return beta - lo_q[:, None], beta - hi_q[:, None]
print("Helpers ready")

## Step 3 · Event effects

All contrasts × outcomes × weights × subgroups × bandwidths × pools, at relative days +7 and +30, for main events, secondary events, negative-control dates and positive controls; β(r) paths, r = 0…30, for the main events.

- Equivalence (TOST): 90% interval [β − q₀.₉₅, β − q₀.₀₅] from the randomization null; margins δ₁ = counterfactual baseline rate (treated mean − β) and δ₂ = one quarter of the Randstorm effect (same outcome and relative day); equivalence if the interval lies inside (−δ, δ).
- Subgroups: balance strata 0.01–1, 1–10, 10–100, ≥100 BTC (`subgroup`); `variant = no_lm` drops the cells of the large-transfer keys from 04c.

In [ ]:
REPORT = ev[ev.event_set.isin(["main", "secondary", "nonquantum_news", "positive_control"])].event.tolist()
MAIN = ev[ev.event_set == "main"].event.tolist()
print("Main events:", MAIN)
diff = TH.xs("diff", level="stat")
lev_T = TH.xs("T", level="stat")
se_s = TH.xs("se", level="stat")
out, pooled, paths = [], [], []
for pool in POOLS:
    for h in BANDWIDTHS:
        B, R = effects(diff, pool, h)
        rep_cols = [c for c in REPORT if c in B.columns]
        p = ri_p(B[rep_cols].to_numpy(), R)
        lo, hi = ri_ci(B[rep_cols].to_numpy(), R)
        Bl, _ = effects(TH.xs("C", level="stat"), pool, h)        # baseline-adjusted control level (negative control)
        for j, e in enumerate(rep_cols):
            df = pd.DataFrame({"event": e, "pool": pool, "h": h, "beta": B[e].values, "p_ri": p[:, j],
                               "ci90_lo": lo[:, j], "ci90_hi": hi[:, j], "theta": diff[e].values,
                               "level_T": lev_T[e].values, "se_sampling": se_s.reindex(diff.index)[e].values,
                               "C_level_beta": Bl.reindex(diff.index)[e].values}, index=diff.index)
            out.append(df.reset_index())
        # pooled test of the 6 main events
        mb = B[MAIN].mean(axis=1).to_numpy()
        null = np.nan_to_num(R) @ DRAWS[pool].T
        p_pool = (1 + np.sum(np.abs(null) >= np.abs(mb)[:, None] - 1e-15, axis=1)) / (1 + null.shape[1])
        q05, q95 = np.quantile(null, 0.05, axis=1), np.quantile(null, 0.95, axis=1)
        pooled.append(pd.DataFrame({"pool": pool, "h": h, "beta_mean": mb, "p_ri": p_pool,
                                    "ci90_lo": mb - q95, "ci90_hi": mb - q05,
                                    "level_T_mean": lev_T[MAIN].mean(axis=1).values}, index=diff.index).reset_index())
        if pool == "main" and h == H_MAIN:
            B_MAIN, R_MAIN = B, R
            bandq = pd.DataFrame({"q05": np.nanquantile(R, .05, axis=1), "q95": np.nanquantile(R, .95, axis=1)}, index=diff.index)
eff = pd.concat(out, ignore_index=True)
eff = eff[eff.rel_day.isin(REPORT_DAYS)]
pooled = pd.concat(pooled, ignore_index=True)
pooled = pooled[pooled.rel_day.isin(REPORT_DAYS)]
ev_meta = ev.set_index("event")[["event_set", "event_date", "strict_ok", "cb_confounded"]]
eff = eff.join(ev_meta, on="event")

# TOST: delta1 = counterfactual baseline rate (level_T - beta); delta2 = 1/4 of the Randstorm effect (RS contrast, same outcome, weight, relative day, variant/subgroup)
rs = eff[(eff.contrast == "RS") & (eff.event == "randstorm")].set_index(["variant", "subgroup", "outcome", "weight", "rel_day", "pool", "h"]).beta
kk = ["variant", "subgroup", "outcome", "weight", "rel_day", "pool", "h"]
for d_ in (eff, pooled):
    d_["delta2"] = d_.set_index(kk).index.map(lambda x: abs(rs.get(x, np.nan)) / 4)
eff["delta1"] = (eff.level_T - eff.beta).abs()
pooled["delta1"] = (pooled.level_T_mean - pooled.beta_mean).abs()
for d_ in (eff, pooled):
    for dn in ["delta1", "delta2"]:
        d_[f"equiv_{dn}"] = (d_.ci90_lo > -d_[dn]) & (d_.ci90_hi < d_[dn])
eff.to_csv("est_20_effects.csv.gz", index=False)
pooled.to_csv("est_20_pooled.csv", index=False)

# main-event paths r = 0...30
paths = B_MAIN[MAIN].join(bandq)
paths.reset_index().to_csv("est_20_paths.csv.gz", index=False)

sel = lambda d_: d_[(d_.contrast == "LC") & (d_.variant == "all") & (d_.subgroup == "all") & (d_.pool == "main") & (d_.h == H_MAIN)]
HEAD = ["c_y1_strict", "c_y1_strict_held", "c_y1_mid", "c_y3", "c_y0", "h_y1s_50"]
t = sel(eff)
t = t[t.outcome.isin(HEAD) & (t.weight == "key") & t.event_set.isin(["main", "nonquantum_news", "secondary"])]
display(t.pivot_table(index=["outcome", "rel_day", "event"], values=["beta", "p_ri", "ci90_lo", "ci90_hi", "se_sampling"], aggfunc="first")
         .reindex(columns=["beta", "ci90_lo", "ci90_hi", "p_ri", "se_sampling"]).round(5))
tp = sel(pooled)
display(tp[tp.outcome.isin(HEAD)].set_index(["outcome", "weight", "rel_day"])[
    ["beta_mean", "ci90_lo", "ci90_hi", "p_ri", "delta1", "delta2", "equiv_delta1", "equiv_delta2"]].round(5))

### Positive and negative controls

- Positive controls (whether the design detects a protective response): Randstorm (`RS`, 2011–15 vs 2016–19) and FTX.
- Negative controls (spurious effects): non-quantum news days (`nonquantum_news`), y0 (outflows to exposed addresses), and the baseline-adjusted control level `C_level_beta`.

In [ ]:
pc = eff[(eff.pool == "main") & (eff.h == H_MAIN) & (eff.variant == "all") & (eff.subgroup == "all")
         & eff.event.isin(["randstorm", "ftx"]) & eff.outcome.isin(["c_y1_strict", "c_y3", "c_y0", "h_y1s_50"])]
display(pc.pivot_table(index=["contrast", "event", "outcome", "weight"], columns="rel_day", values=["beta", "p_ri"]).round(5))
nc = eff[(eff.pool == "main") & (eff.h == H_MAIN) & (eff.variant == "all") & (eff.subgroup == "all") & (eff.contrast == "LC")
         & (eff.weight == "key") & (eff.rel_day == 30)]
display(nc[nc.event_set == "nonquantum_news"][["event", "outcome", "beta", "p_ri"]].pivot(index="outcome", columns="event").round(5))
display(nc[nc.event.isin(MAIN) & (nc.outcome == "c_y1_strict")][["event", "C_level_beta"]].round(5))

## Step 4 · Joint Google × BIP-361 estimation

Daily L−C differences of the two panels (`d_*`, baseline removed) stacked into 62 observations; each event has effects for days 0–7 and 8–30, both applying on overlapping calendar days. Identification comes from days unique to Google (03-31…04-13) and to BIP-361 (05-01…05-14); the Google effect is 0 after its day +30. Null: pairs of placebo dates (29 × 28 = 812) in the same design. `implied_cum` = 8 × first segment + 23 × second segment (cumulative effect at +30).

In [ ]:
G, Bv = JOINT
gap = (ev.set_index("event").event_date[Bv] - ev.set_index("event").event_date[G]).days
def design():
    X = []
    for panel in (G, Bv):
        for r in range(0, 31):
            kG = r if panel == G else r + gap
            kB = r - gap if panel == G else r
            row = []
            for a, b in SEGMENTS:
                row.append(1.0 if a <= kG <= b else 0.0)
            for a, b in SEGMENTS:
                row.append(1.0 if a <= kB <= b else 0.0)
            X.append(row)
    return np.array(X)
X = design()
names = [f"{e}_{a}-{b}" for e in (G, Bv) for a, b in SEGMENTS]
assert np.linalg.matrix_rank(X) == X.shape[1], "design matrix is rank-deficient"
XtXi = np.linalg.pinv(X)
jrows = []
pool_ev = POOL_EVENTS["main"]
pairs = [(i, j) for i in range(len(pool_ev)) for j in range(len(pool_ev)) if i != j][:N_PAIRS_MAX]
for variant in ["all", "no_lm"]:
    for o in OUT_D:
        for wt in ["key", "btc"]:
            key_ = ("LC", variant, "all", f"d_{o}", wt)
            path = B_MAIN.loc[key_]                       # relative day x event
            y = np.concatenate([path[G].reindex(range(31)).values, path[Bv].reindex(range(31)).values])
            if np.isnan(y).any():
                continue
            b = XtXi @ y
            Rp = path[pool_ev].reindex(range(31)).to_numpy()
            nulls = np.array([XtXi @ np.concatenate([Rp[:, i], Rp[:, j]]) for i, j in pairs])
            nulls = nulls[~np.isnan(nulls).any(1)]
            for c, nm in enumerate(names):
                jrows.append({"variant": variant, "outcome": f"d_{o}", "weight": wt, "term": nm, "beta": b[c],
                              "p_ri": (1 + np.sum(np.abs(nulls[:, c]) >= abs(b[c]))) / (1 + len(nulls)),
                              "null_q05": np.quantile(nulls[:, c], .05), "null_q95": np.quantile(nulls[:, c], .95)})
            for e in (G, Bv):
                w = np.array([(b_ - a + 1) if nm.startswith(e) else 0 for nm, (a, b_) in zip(names, SEGMENTS * 2)])
                val, nv = w @ b, nulls @ w
                jrows.append({"variant": variant, "outcome": f"d_{o}", "weight": wt, "term": f"{e}_implied_cum30",
                              "beta": val, "p_ri": (1 + np.sum(np.abs(nv) >= abs(val))) / (1 + len(nv)),
                              "null_q05": np.quantile(nv, .05), "null_q95": np.quantile(nv, .95)})
joint = pd.DataFrame(jrows)
joint.to_csv("est_20_joint.csv", index=False)
print(f"Joint estimation: {len(pairs)} pairs of placebo paths as the null distribution; Google and BIP-361 are {gap} days apart")
display(joint[(joint.variant == "all") & (joint.weight == "key")].pivot(index="term", columns="outcome", values=["beta", "p_ri"]).round(5))

## Step 5 · Stratified piecewise-exponential model

Per date × contrast × interval (0–7, 8–30, 0–30) × hazard event (cumulative y1 strict ≥ 50%; also > 0, held coins ≥ 50%, y0 ≥ 50%): O ~ Poisson(exp(α_s + β·treated) · R), with CEM cells s as strata and R = key-days at risk. With stratum effects profiled out, β solves Σ_s [O_T,s − (O_T,s + O_C,s) · e^β R_T,s / (e^β R_T,s + R_C,s)] = 0 (one-dimensional root search). Baseline removal and inference as above.

In [ ]:
hz_cols = ",\n    ".join(f"SUM(IF(arm = 'T', h_{h}_n, 0)) AS OT_{h}, SUM(IF(arm = 'T', h_{h}_risk, 0)) AS RT_{h}, "
                         f"SUM(IF(arm = 'C', h_{h}_n, 0)) AS OC_{h}, SUM(IF(arm = 'C', h_{h}_risk, 0)) AS RC_{h}" for h in HZ)
iv_case = " ".join(f"WHEN rel_day BETWEEN {a} AND {b} THEN '{n}'" for n, (a, b) in HR_INTERVALS.items() if n != "0-30")
SQL_HR = f'''
WITH pe AS (
  SELECT event, grp, cem_id, is_large_mover, rel_day, bal_bin, age_bin, family, fs_bin, rec_bin, nutxo_bin,
         {", ".join(f"h_{h}_n, h_{h}_risk" for h in HZ)}
  FROM `{DS}.panel_event`
),
u AS (
  SELECT pe.*, x.contrast, x.arm, x.stratum
  FROM pe,
  {UNNEST_CONTRASTS}
  WHERE x.arm IS NOT NULL
),
iv AS (
  SELECT u.*, i AS intv FROM u, UNNEST(['0-30', CASE {iv_case} END]) AS i
)
SELECT event, contrast, intv, stratum,
    {hz_cols}
FROM iv
GROUP BY event, contrast, intv, stratum
HAVING {" OR ".join(f"(SUM(h_{h}_n) > 0)" for h in HZ)}
'''
hr_raw = q(SQL_HR)
def strat_loghr(OT, RT, OC, RC):
    m = (RT > 0) & (RC > 0) & (OT + OC > 0)
    OT, RT, OC, RC = OT[m], RT[m], OC[m], RC[m]
    if OT.sum() == 0 or OC.sum() == 0:
        return np.nan
    g = lambda b: np.sum(OT - (OT + OC) * np.exp(b) * RT / (np.exp(b) * RT + RC))
    return brentq(g, -20, 20)
hr_rows = []
for (e, c, iv), g in hr_raw.groupby(["event", "contrast", "intv"]):
    for h in HZ:
        hr_rows.append({"event": e, "contrast": c, "intv": iv, "hazard": h,
                        "loghr": strat_loghr(*(g[f"{p}_{h}"].to_numpy(float) for p in ["OT", "RT", "OC", "RC"])),
                        "O_T": g[f"OT_{h}"].sum(), "O_C": g[f"OC_{h}"].sum()})
hr = pd.DataFrame(hr_rows)
HR = hr.pivot_table(index=["contrast", "hazard", "intv"], columns="event", values="loghr", aggfunc="first").reindex(columns=ev.event.tolist())
hout = []
for pool in POOLS:
    for h in BANDWIDTHS:
        B, R = effects(HR, pool, h)
        cols = [c for c in REPORT if c in B.columns]
        p = ri_p(B[cols].to_numpy(), R)
        for j, e in enumerate(cols):
            hout.append(pd.DataFrame({"event": e, "pool": pool, "h": h, "loghr": HR[e].values, "beta": B[e].values,
                                      "p_ri": p[:, j]}, index=HR.index).reset_index())
        mb = B[MAIN].mean(axis=1).to_numpy()
        null = np.nan_to_num(R) @ DRAWS[pool].T
        pp = (1 + np.sum(np.abs(null) >= np.abs(mb)[:, None], axis=1)) / (1 + null.shape[1])
        hout.append(pd.DataFrame({"event": "POOLED_MAIN", "pool": pool, "h": h, "loghr": np.nan, "beta": mb, "p_ri": pp},
                                 index=HR.index).reset_index())
hz_eff = pd.concat(hout, ignore_index=True).merge(hr, on=["event", "contrast", "hazard", "intv"], how="left", suffixes=("", "_raw"))
hz_eff.to_csv("est_20_hazard.csv", index=False)
v = hz_eff[(hz_eff.pool == "main") & (hz_eff.h == H_MAIN) & (hz_eff.contrast == "LC") & (hz_eff.hazard == "y1s_50")]
display(v.pivot_table(index="event", columns="intv", values=["loghr", "beta", "p_ri"]).round(4))

## Step 6 · Figures

1. `fig_20_effects.png`: β at +30 (LC, key- and BTC-weighted; y1 strict, y1 strict held, y0); grey = placebo leave-one-out residuals, red = main events, blue = negative-control dates.
2. `fig_20_paths.png`: β(r) of the main events, r = 0…30, with the 5–95% band of placebo residuals.
3. `fig_20_km.png`: cumulative incidence of first de-risking (y1 strict ≥ 50%) for the main events, L and CEM-weighted C.
4. `fig_20_hetero.png`: β at +30 by balance stratum.

In [ ]:
import matplotlib.pyplot as plt
base = eff[(eff.pool == "main") & (eff.h == H_MAIN) & (eff.contrast == "LC") & (eff.variant == "all") & (eff.subgroup == "all") & (eff.rel_day == 30)]
panels = [("c_y1_strict", "key"), ("c_y1_strict", "btc"), ("c_y1_strict_held", "key"), ("c_y0", "key")]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
idx_pool = [EV_IDX[e] for e in POOL_EVENTS["main"]]
for ax, (o, w) in zip(axes, panels):
    key_ = ("LC", "all", "all", o, w, 30)
    r = B_MAIN.loc[key_, POOL_EVENTS["main"]]
    ax.scatter(ev.event_date.iloc[idx_pool], r.values, color="grey", s=14, label="placebo (LOO residual)")
    b = base[(base.outcome == o) & (base.weight == w)]
    for s_, col in [("main", "C3"), ("nonquantum_news", "C0"), ("secondary", "C2")]:
        bb = b[b.event_set == s_]
        ax.scatter(bb.event_date, bb.beta, color=col, s=36, label=s_)
        for rr in bb.itertuples():
            ax.annotate(f"{rr.event}\np={rr.p_ri:.2f}", (rr.event_date, rr.beta), fontsize=6, xytext=(3, 2), textcoords="offset points")
    ax.axhline(0, color="k", lw=0.6); ax.set_title(f"{o} ({w}), day +30", fontsize=9)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
axes[0].legend(fontsize=6)
plt.tight_layout(); plt.savefig("fig_20_effects.png", dpi=130); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
for ax, e in zip(axes.flat, MAIN):
    for o, col in [("c_y1_strict", "C3"), ("c_y1_strict_held", "C1")]:
        key_ = ("LC", "all", "all", o, "key")
        s = paths.loc[key_]
        ax.plot(s.index, s[e], color=col, label=o)
        ax.fill_between(s.index, s.q05, s.q95, color=col, alpha=0.12)
    ax.axhline(0, color="k", lw=0.6); ax.set_title(e, fontsize=9)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig_20_paths.png", dpi=130); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
for ax, e in zip(axes.flat, MAIN):
    for stat, col, lab in [("T", "C3", "L"), ("C", "C0", "C (CEM-weighted)")]:
        s = TH.loc[("LC", "all", "all", "h_y1s_50", "key", stat)][e]
        ax.plot(s.index, s.values, color=col, label=lab)
    ax.set_title(f"{e}: first de-risking (y1 strict ≥ 50%)", fontsize=9)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig_20_km.png", dpi=130); plt.show()

het = eff[(eff.pool == "main") & (eff.h == H_MAIN) & (eff.contrast == "LC") & (eff.variant == "all") & (eff.rel_day == 30)
          & (eff.outcome == "c_y1_strict") & (eff.weight == "key") & eff.event.isin(MAIN)]
fig, ax = plt.subplots(figsize=(9, 4))
sgs = ["0.01-1", "1-10", "10-100", "100+"]
for k_, sg in enumerate(sgs):
    hh = het[het.subgroup == sg].set_index("event").reindex(MAIN)
    ax.errorbar(np.arange(len(MAIN)) + (k_ - 1.5) * 0.18, hh.beta, yerr=[hh.beta - hh.ci90_lo, hh.ci90_hi - hh.beta],
                fmt="o", ms=4, capsize=2, label=f"{sg} BTC")
ax.set_xticks(range(len(MAIN))); ax.set_xticklabels(MAIN); ax.axhline(0, color="k", lw=0.6)
ax.set_title("LC, cumulative y1 strict share, day +30, by balance tier (RI 90% CI)", fontsize=9); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig_20_hetero.png", dpi=130); plt.show()

In [ ]:
def rec(df_):
    return df_.astype(str).to_dict(orient="records")
head_eff = sel(eff)
head_eff = head_eff[head_eff.outcome.isin(HEAD) & head_eff.event_set.isin(["main", "nonquantum_news", "secondary", "positive_control"])]
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "config": {"h_main": H_MAIN, "bandwidths": BANDWIDTHS, "pools": POOLS, "report_days": REPORT_DAYS,
                     "joint": JOINT, "segments": SEGMENTS, "contrasts": CONTRASTS, "hazards": HZ},
          "bq_usd_estimate": round(SPENT["usd"], 3),
          "pool_sizes": {p: len(v) for p, v in POOL_EVENTS.items()},
          "headline_effects": rec(head_eff[["event", "outcome", "weight", "rel_day", "beta", "ci90_lo", "ci90_hi", "p_ri",
                                            "se_sampling", "delta1", "delta2", "equiv_delta1", "equiv_delta2"]]),
          "pooled_main": rec(sel(pooled)[sel(pooled).outcome.isin(HEAD)]),
          "joint": rec(joint[(joint.variant == "all")]),
          "hazard_LC": rec(hz_eff[(hz_eff.pool == "main") & (hz_eff.h == H_MAIN) & (hz_eff.contrast == "LC")])}
with open("build_report_20.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    for fn in ["build_report_20.json", "est_20_effects.csv.gz", "est_20_pooled.csv", "est_20_joint.csv", "est_20_hazard.csv",
               "est_20_paths.csv.gz", "fig_20_effects.png", "fig_20_paths.png", "fig_20_km.png", "fig_20_hetero.png"]:
        if os.path.exists(fn): files.download(fn)
except Exception:
    print("Report saved to build_report_20.json")